In [2]:
pip install plotly

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.9 MB 856.6 kB/s eta 0:00:11
   ---- ----------------------------------- 1.0/9.9 MB 1.0 MB/s eta 0:00:09
   ----- ---------------------------------- 1.3/9.9 MB 1.2 MB/s eta 0:00:08
   ------- -------------------------------- 1.8/9.9 MB 1.3 MB/s eta 0:00:07
   --------- ------------------------------ 2.4/9.9 MB 1.5 MB/s eta 0:00:06
   ------------ --------------------------- 3.1/9.9 MB 1.8 MB/s eta 0:00:04
   ------------ --------------------------- 3.1/9.9 MB 1.8 MB/s eta 0:00:04
   --------------- ------------------------ 3.9/9.9 MB 1.8 MB/s eta 0:00:04
   ----------------- ---------------------- 4

In [44]:
#libs
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.offline import init_notebook_mode

init_notebook_mode(connected=True)

In [24]:
df1 = pd.read_csv("data/nykaa_campaign_data.csv")
df2 = pd.read_csv("data/purplle_campaign_data.csv")
df3 = pd.read_csv("data/tira_campaign_data.csv")

In [25]:
df1["source"] = "Nika"
df2["source"] = "Purplle"
df3["source"] = "Tira"

In [45]:
df = pd.concat([df1, df2, df3], ignore_index=True)

In [46]:
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

# data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 166665 entries, 0 to 166664
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   campaign_id       166665 non-null  str    
 1   campaign_type     166665 non-null  str    
 2   target_audience   166665 non-null  str    
 3   duration          166665 non-null  int64  
 4   channel_used      166665 non-null  str    
 5   impressions       166665 non-null  int64  
 6   clicks            166665 non-null  int64  
 7   leads             166665 non-null  int64  
 8   conversions       166665 non-null  int64  
 9   revenue           166665 non-null  int64  
 10  acquisition_cost  166665 non-null  float64
 11  roi               166665 non-null  float64
 12  language          166665 non-null  str    
 13  engagement_score  166665 non-null  float64
 14  customer_segment  166665 non-null  str    
 15  date              166665 non-null  str    
 16  source            166665 non-nu

In [28]:
df["channel_list"] = df["channel_used"].str.split(",")

df["channel_list"] = df["channel_list"].apply(
    lambda x: [i.strip().lower() for i in x]
)

In [29]:
#unique channels
all_channels = set([ch for sublist in df["channel_list"] for ch in sublist])

for ch in all_channels:
    df[f"channel_{ch}"] = df["channel_list"].apply(lambda x: int(ch in x))

In [30]:
df.head()

,campaign_id,campaign_type,target_audience,duration,channel_used,impressions,clicks,leads,conversions,revenue,...,customer_segment,date,source,channel_list,channel_email,channel_youtube,channel_facebook,channel_google,channel_whatsapp,channel_instagram
0,NY-CMP-1000,Social Media,College Students,21,"WhatsApp, YouTube",57804,6156,3616,2355,1867515,...,College Students,29-04-2025,Nika,"[whatsapp, youtube]",0,1,0,0,1,0
1,NY-CMP-1001,Paid Ads,Tier 2 City Customers,18,YouTube,91801,3321,1971,1357,1046247,...,College Students,06-04-2025,Nika,[youtube],0,1,0,0,0,0
2,NY-CMP-1002,Influencer,Youth,23,"WhatsApp, Google, YouTube",15536,2182,952,755,197055,...,College Students,14-01-2025,Nika,"[whatsapp, google, youtube]",0,1,0,1,1,0
3,NY-CMP-1003,Email,Working Women,18,"YouTube, Facebook, Instagram",88114,8413,2231,947,376906,...,College Students,04-06-2025,Nika,"[youtube, facebook, instagram]",0,1,1,0,0,1
4,NY-CMP-1004,Paid Ads,College Students,10,"Facebook, Instagram",96871,3743,2060,1258,518296,...,Tier 2 City Customers,29-12-2024,Nika,"[facebook, instagram]",0,0,1,0,0,1


In [31]:
df["ctr"] = df["clicks"] / df["impressions"].replace(0, np.nan)
df["lead_rate"] = df["leads"] / df["clicks"].replace(0, np.nan)
df["conversion_rate"] = df["conversions"] / df["leads"].replace(0, np.nan)

# Cost efficiency
df["cost_per_conversion"] = df["acquisition_cost"] / df["conversions"].replace(0, np.nan)

In [32]:
df.describe()

,duration,impressions,clicks,leads,conversions,revenue,acquisition_cost,roi,engagement_score,channel_email,channel_youtube,channel_facebook,channel_google,channel_whatsapp,channel_instagram,ctr,lead_rate,conversion_rate,cost_per_conversion
count,166665.000000,166665.000000,166665.000000,166665.000000,166665.000000,1.666650e+05,166665.000000,166665.000000,166665.000000,166665.000000,166665.000000,166665.000000,166665.000000,166665.000000,166665.000000,166665.000000,166665.000000,166665.000000,166665.000000
mean,17.490367,55060.854091,4682.367546,1871.514181,1029.088885,5.139066e+05,376.094881,2.690567,13.766293,0.333543,0.333501,0.333471,0.333459,0.332733,0.333597,0.085023,0.399321,0.549601,2.076187
std,7.504843,25970.653994,3176.502340,1429.590998,858.011218,4.876443e+05,534.939555,4.481216,6.330005,0.471480,0.471465,0.471455,0.471450,0.471193,0.471499,0.037538,0.115440,0.144450,10.360204
min,5.000000,10001.000000,202.000000,48.000000,17.000000,3.895000e+03,8.180000,-0.990000,2.560000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.019959,0.197995,0.288136,0.001330
25%,11.000000,32566.000000,2109.000000,779.000000,401.000000,1.776610e+05,106.730000,0.040000,8.380000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.052561,0.299494,0.424242,0.077808
50%,17.000000,55110.000000,3904.000000,1476.000000,776.000000,3.592500e+05,208.510000,1.230000,13.590000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.084995,0.398932,0.550072,0.264315
75%,24.000000,77574.000000,6688.000000,2598.000000,1404.000000,6.847970e+05,427.570000,3.580000,18.790000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.117705,0.499495,0.674476,1.027215
max,30.000000,100000.000000,14944.000000,8876.000000,6686.000000,4.579910e+06,15473.160000,79.300000,30.990000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.149995,0.599981,0.799931,870.730000


Revenue by channel

In [34]:
df_exploded = df.explode("channel_list")
df_exploded["channel_list"] = df_exploded["channel_list"].str.strip().str.lower()
rev_df = (
    df_exploded
    .groupby("channel_list")["revenue"]
    .sum()
    .reset_index()
    .sort_values("revenue", ascending=True)
)
fig = px.bar(
    rev_df.sort_values("revenue", ascending=False),
    x="revenue",
    y="channel_list",
    orientation="h",
    title="Revenue by Marketing Channel"
)
fig.show()

In [48]:
fig = px.bar(
    rev_df.sort_values("revenue", ascending=True),
    x="revenue",
    y="channel_list",
    orientation="h",
    title="Revenue by Channel (Zoomed View)"
)
fig.update_xaxes(range=[2e10, 3e10])

fig.show()

In [49]:
funnel_data = {
    "stage": ["Impressions", "Clicks", "Leads", "Conversions"],
    "value": [
        df["impressions"].sum(),
        df["clicks"].sum(),
        df["leads"].sum(),
        df["conversions"].sum()
    ]
}

funnel_df = pd.DataFrame(funnel_data)
funnel_df

,stage,value
0,Impressions,9176717247
1,Clicks,780386787
2,Leads,311915911
3,Conversions,171513099


In [37]:
fig = go.Figure(go.Funnel(
    y=funnel_df["stage"],
    x=funnel_df["value"],
    textinfo="value+percent previous"
))

fig.update_layout(
    title="Marketing Funnel: From Impressions to Conversions"
)

fig.show()

In [38]:
impr = df["impressions"].sum()
clicks = df["clicks"].sum()
leads = df["leads"].sum()
conv = df["conversions"].sum()

print("CTR:", clicks / impr)
print("Lead Rate:", leads / clicks)
print("Conversion Rate:", conv / leads)

CTR: 0.08503986403799459
Lead Rate: 0.3996939930250254
Conversion Rate: 0.5498696698418825


In [39]:
df_exploded = df.explode("channel_list")

funnel_channel = (
    df_exploded
    .groupby("channel_list")[["impressions", "clicks", "leads", "conversions"]]
    .sum()
    .reset_index()
)

funnel_channel.head()

,channel_list,impressions,clicks,leads,conversions
0,email,3059662952,261008477,104477930,57483158
1,facebook,3068497354,260667815,104147736,57237666
2,google,3052666418,258564394,103235186,56731745
3,instagram,3062330473,261025997,104498011,57484941
4,whatsapp,3050576527,259751314,103816518,57077895


In [40]:
channels_to_plot = ["email", "facebook", "google", "instagram", "whatsapp"]
df_exploded = df.explode("channel_list")

funnel_channel = (
    df_exploded
    .groupby("channel_list")[["impressions", "clicks", "leads", "conversions"]]
    .sum()
    .reset_index()
)

for channel_name in channels_to_plot:
    data = funnel_channel[funnel_channel["channel_list"] == channel_name]
    if data.empty:
        continue

    fig = go.Figure(go.Funnel(
        y=["Impressions", "Clicks", "Leads", "Conversions"],
        x=[
            data["impressions"].values[0],
            data["clicks"].values[0],
            data["leads"].values[0],
            data["conversions"].values[0]
        ],
        textinfo="value+percent previous"
    ))

    fig.update_layout(
        title=f"Funnel for {channel_name.title()} Channel"
    )

    fig.show()

In [41]:
df_exploded = df.explode("channel_list")
metrics_channel = (
    df_exploded.groupby("channel_list")
    .agg(
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        leads=("leads", "sum"),
        conversions=("conversions", "sum"),
        revenue=("revenue", "sum"),
        acquisition_cost=("acquisition_cost", "sum")
    )
    .reset_index()
)
metrics_channel["CTR"] = metrics_channel["clicks"] / metrics_channel["impressions"]
metrics_channel["Lead_Rate"] = metrics_channel["leads"] / metrics_channel["clicks"]
metrics_channel["Conversion_Rate"] = metrics_channel["conversions"] / metrics_channel["leads"]
metrics_channel["ROI"] = metrics_channel["revenue"] / metrics_channel["acquisition_cost"]

metrics_channel

,channel_list,impressions,clicks,leads,conversions,revenue,acquisition_cost,CTR,Lead_Rate,Conversion_Rate,ROI
0,email,3059662952,261008477,104477930,57483158,28701227780,20733459.44,0.085306,0.400286,0.550194,1384.295171
1,facebook,3068497354,260667815,104147736,57237666,28527814614,20895827.38,0.084950,0.399542,0.549581,1365.239772
2,google,3052666418,258564394,103235186,56731745,28430612497,20904447.83,0.084701,0.399263,0.549539,1360.026954
3,instagram,3062330473,261025997,104498011,57484941,28759625581,20790646.66,0.085238,0.400336,0.550106,1383.296347
4,whatsapp,3050576527,259751314,103816518,57077895,28469625887,20941885.00,0.085148,0.399677,0.549796,1359.458611
5,youtube,3067659282,260336538,103889827,57086079,28471946681,21038138.30,0.084865,0.399060,0.549487,1353.349155


In [42]:
fig = px.bar(
    metrics_channel.sort_values("Conversion_Rate", ascending=True),
    x="Conversion_Rate",
    y="channel_list",
    orientation="h",
    title="Conversion Rate by Channel",
    text=metrics_channel["Conversion_Rate"].apply(lambda x: f"{x:.2%}")
)

fig.update_traces(textposition="outside")
fig.show()

In [43]:
fig = px.bar(
    metrics_channel.sort_values("ROI", ascending=True),
    x="ROI",
    y="channel_list",
    orientation="h",
    title="ROI by Channel",
    text=metrics_channel["ROI"].apply(lambda x: f"{x:.2f}")
)

fig.update_traces(textposition="outside")
fig.show()